# Example 13 — Taylor–Green vortex: unsteady Navier–Stokes with an exact solution

The one exact unsteady incompressible Navier–Stokes solution everyone uses for
validation: an array of counter-rotating vortices that decay by viscosity,
$$u = -\cos x\,\sin y\;e^{-2\nu t},\qquad v = \sin x\,\cos y\;e^{-2\nu t},\qquad p = -\tfrac{1}{4}(\cos 2x + \cos 2y)\,e^{-4\nu t},$$
on $[0,2\pi]^2$, here with $\nu = 0.1$, $t\in[0,1]$.

**PINN design — three ideas worth teaching:**
1. **Full NS as residuals:** outputs $(u,v,p)$; the loss stacks two momentum residuals and
   the continuity equation. Pressure is an unknown *field* now (vs the constant $G$ of Ex. 12).
2. **Hard periodicity:** feed $(\sin x, \cos x, \sin y, \cos y, t)$ — the network *cannot*
   represent a non-periodic field. No boundary loss at all.
3. **Pressure gauge:** with periodic BCs, $p$ is defined only up to a constant — fix it with
   a zero-mean penalty $(\bar{p})^2$.

**Verified on an AMD Instinct MI210 (ROCm PyTorch):** rel. L2 at $t{=}1$: **u 0.0054, v 0.0061**;
kinetic-energy decay $E(1)/E_{exact}(1) = 1.0005$; training 546 s (15k epochs).
On Colab GPU expect a similar few-minute run; CPU ~30–40 min.

In [ ]:
# Cell 1 -- Setup: exact solution, network with hard-periodic embedding
import time
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

NU, T, PI = 0.1, 1.0, np.pi

def exact(x, y, t):
    E = torch.exp(-2*NU*t)
    u = -torch.cos(x)*torch.sin(y)*E
    v =  torch.sin(x)*torch.cos(y)*E
    p = -0.25*(torch.cos(2*x)+torch.cos(2*y))*torch.exp(-4*NU*t)
    return u, v, p

net = nn.Sequential(nn.Linear(5, 96), nn.Tanh(), nn.Linear(96, 96), nn.Tanh(),
                    nn.Linear(96, 96), nn.Tanh(), nn.Linear(96, 96), nn.Tanh(),
                    nn.Linear(96, 3)).to(device)

def uvp(x, y, t):
    z = torch.cat([torch.sin(x), torch.cos(x), torch.sin(y), torch.cos(y), t], 1)
    o = net(z)
    return o[:, 0:1], o[:, 1:2], o[:, 2:3]

def grads(f, *xs):
    return [torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0] for x in xs]

In [ ]:
# Cell 2 -- Train: momentum-x, momentum-y, continuity, IC, pressure gauge
EPOCHS = 15000
opt = torch.optim.Adam(net.parameters(), 1e-3)
t0 = time.perf_counter()
for e in range(EPOCHS):
    if e == 10000:
        for g in opt.param_groups: g['lr'] = 2e-4
    if e == 13000:
        for g in opt.param_groups: g['lr'] = 5e-5
    opt.zero_grad()
    N = 4096
    x = (torch.rand(N,1,device=device)*2*PI).requires_grad_(True)
    y = (torch.rand(N,1,device=device)*2*PI).requires_grad_(True)
    t = (torch.rand(N,1,device=device)*T).requires_grad_(True)
    u, v, p = uvp(x, y, t)
    ux, uy, ut = grads(u, x, y, t)
    vx, vy, vt = grads(v, x, y, t)
    px, py = grads(p, x, y)
    uxx = grads(ux, x)[0]; uyy = grads(uy, y)[0]
    vxx = grads(vx, x)[0]; vyy = grads(vy, y)[0]
    rx = ut + u*ux + v*uy + px - NU*(uxx+uyy)     # x-momentum
    ry = vt + u*vx + v*vy + py - NU*(vxx+vyy)     # y-momentum
    rc = ux + vy                                   # continuity
    xi = torch.rand(1024,1,device=device)*2*PI; yi = torch.rand(1024,1,device=device)*2*PI
    ti = torch.zeros_like(xi)
    ue, ve, _ = exact(xi, yi, ti)
    ui, vi, _ = uvp(xi, yi, ti)
    loss = (rx**2).mean() + (ry**2).mean() + (rc**2).mean() \
         + 10*((ui-ue)**2).mean() + 10*((vi-ve)**2).mean() \
         + (p.mean())**2                           # pressure gauge
    loss.backward(); opt.step()
    if e % 3000 == 0: print(f'epoch {e:6d}  loss {loss.item():.2e}  ({time.perf_counter()-t0:.0f}s)')
if device.type == 'cuda': torch.cuda.synchronize()
print(f'\ntraining: {time.perf_counter()-t0:.0f} s')

In [ ]:
# Cell 3 -- Validate: fields at t=1, and the kinetic-energy decay law
n = 128
xs = torch.linspace(0, 2*PI, n, device=device)
X, Y = torch.meshgrid(xs, xs, indexing='ij')
Xf = X.reshape(-1,1); Yf = Y.reshape(-1,1)

def fields(tval):
    tt = torch.full_like(Xf, tval)
    with torch.no_grad(): u, v, p = uvp(Xf, Yf, tt)
    ue, ve, pe = exact(Xf, Yf, tt)
    return [a.reshape(n,n).cpu().numpy() for a in (u, v, ue, ve)]

u1, v1, ue1, ve1 = fields(1.0)
err_u = np.sqrt(np.mean((u1-ue1)**2)/np.mean(ue1**2))
err_v = np.sqrt(np.mean((v1-ve1)**2)/np.mean(ve1**2))
print(f'rel L2 at t=1:  u {err_u:.4f},  v {err_v:.4f}')

ts = np.linspace(0, 1, 21); E_p, E_e = [], []
for tv in ts:
    u, v, ue, ve = fields(float(tv))
    E_p.append(0.5*np.mean(u**2+v**2)); E_e.append(0.5*np.mean(ue**2+ve**2))

h = 2*PI/(n-1)
def vort(u, v):
    w = np.zeros_like(u)
    w[1:-1,1:-1] = (v[2:,1:-1]-v[:-2,1:-1])/(2*h) - (u[1:-1,2:]-u[1:-1,:-2])/(2*h)
    return w

fig, ax = plt.subplots(1, 3, figsize=(14.5, 4.2))
c0 = ax[0].contourf(X.cpu(), Y.cpu(), vort(ue1, ve1), 21, cmap='RdBu_r'); plt.colorbar(c0, ax=ax[0])
ax[0].set_title('exact vorticity @ t=1'); ax[0].set_xlabel('x'); ax[0].set_ylabel('y')
c1 = ax[1].contourf(X.cpu(), Y.cpu(), vort(u1, v1), 21, cmap='RdBu_r'); plt.colorbar(c1, ax=ax[1])
ax[1].set_title(f'PINN vorticity @ t=1  (rel L2(u)={err_u:.3f})'); ax[1].set_xlabel('x')
ax[2].semilogy(ts, E_e, 'g', lw=2.2, label='exact  $E_0 e^{-4\\nu t}$')
ax[2].semilogy(ts, E_p, 'r--', lw=1.6, label='PINN')
ax[2].set_xlabel('t'); ax[2].set_ylabel('kinetic energy'); ax[2].legend(fontsize=9)
ax[2].set_title('Viscous decay of kinetic energy'); ax[2].grid(alpha=.3, which='both')
plt.tight_layout(); plt.show()
print(f'E(1)/E_exact(1) = {E_p[-1]/E_e[-1]:.4f}   (decay law e^(-4·nu·t) reproduced)')

## Observations (for fluid-dynamics notes)

- **A full unsteady NS solver in ~60 lines.** Two momentum residuals + continuity + IC —
  no projection method, no staggered grid, no Poisson solve for pressure. The
  incompressibility that costs classical CFD its pressure-correction machinery is just a
  third residual here.
- **Hard periodicity kills a whole class of loss terms.** The $(\sin,\cos)$ embedding is to
  space what Example 17's trick is to time — no periodic-BC penalties, nothing to balance.
- **Pressure needs a gauge.** With no pressure BC anywhere, $p$ floats by a constant;
  the $(\bar{p})^2$ penalty pins it. Forgetting this is the classic first bug in NS PINNs
  (Example 14 uses a point anchor instead — either works).
- **Validate physics, not just fields:** the energy plot shows the PINN reproducing the
  $e^{-4\nu t}$ decay *law* to 0.05% — a stronger statement than pointwise error, and the
  kind of check CFD papers demand.
- **Verified on AMD MI210 (ROCm):** u/v rel-L2 ≈ 0.5–0.6% at t=1 after 15k epochs (546 s).
  PyTorch's `cuda` API is the ROCm backend — the code is identical on NVIDIA/AMD.

**Experiments to try:** raise ν and watch the decay steepen; drop the continuity residual
and watch mass leak; replace $(u,v,p)$ with a streamfunction ($u=\psi_y, v=-\psi_x$ —
continuity exact by construction, at third-derivative cost); add 4 noisy 'PIV' snapshots
and recover ν (inverse, Example 3's pattern).